# RetainIQ — Phase 4.2: Hypothesis Testing & Churn Driver Analysis

## Objective

I move from descriptive EDA to formal statistical testing.

I will use Welch's independent-samples t-test for selected numeric variables and chi-square tests
for selected categorical variables.

I will separate association from causation and review potential leakage before modeling.

## 1. Connect to MySQL

In [1]:
from getpass import getpass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mysql.connector
from mysql.connector import Error
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

MYSQL_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "retainiq_user",
    "password": getpass("Enter MySQL password for retainiq_user: "),
    "database": "retainiq"
}

def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)

def run_query(query, params=None):
    connection = None
    cursor = None
    try:
        connection = get_connection()
        cursor = connection.cursor(dictionary=True)
        cursor.execute(query, params or ())
        return pd.DataFrame(cursor.fetchall())
    except Error as exc:
        raise RuntimeError(f"MySQL query failed: {exc}") from exc
    finally:
        if cursor:
            cursor.close()
        if connection and connection.is_connected():
            connection.close()

print("MySQL analytical connection is ready.")

MySQL analytical connection is ready.


## 2. Build the Statistical Testing Population

In [2]:
query = '''
SELECT
    f.customer_id,
    f.tenure_months,
    f.monthly_charge,
    f.total_revenue,
    f.satisfaction_score,
    f.cltv,
    f.churn_label,
    d.gender,
    a.contract,
    a.payment_method,
    s.internet_type,
    s.premium_tech_support
FROM fact_customer_status f
JOIN dim_demographics d ON f.customer_id = d.customer_id
JOIN dim_account a ON f.customer_id = a.customer_id
JOIN dim_services s ON f.customer_id = s.customer_id
WHERE f.customer_status IN ('Stayed', 'Churned');
'''

stats_df = run_query(query)
stats_df["churn_flag"] = (stats_df["churn_label"] == "Yes").astype(int)

print(f"Testing population: {len(stats_df):,}")

Testing population: 6,589


In [5]:
# MySQL DECIMAL fields can arrive in pandas as object/Decimal.
# I convert the variables used in statistical testing to numeric types.

numeric_columns = [
    "tenure_months",
    "monthly_charge",
    "total_revenue",
    "satisfaction_score",
    "cltv"
]

for column in numeric_columns:
    stats_df[column] = pd.to_numeric(
        stats_df[column],
        errors="coerce"
    )

print("Numeric columns converted successfully.")
print(stats_df[numeric_columns].dtypes)

Numeric columns converted successfully.
tenure_months           int64
monthly_charge        float64
total_revenue         float64
satisfaction_score      int64
cltv                    int64
dtype: object


## 3. Numeric Test Function

In [7]:
def numeric_test(data, column):
    numeric_data = pd.to_numeric(
        data[column],
        errors="coerce"
    )

    churned = numeric_data.loc[
        data["churn_flag"] == 1
    ].dropna()

    retained = numeric_data.loc[
        data["churn_flag"] == 0
    ].dropna()

    statistic, p_value = stats.ttest_ind(
        churned.astype(float),
        retained.astype(float),
        equal_var=False
    )

    return {
        "variable": column,
        "churned_mean": churned.mean(),
        "retained_mean": retained.mean(),
        "difference": churned.mean() - retained.mean(),
        "t_statistic": statistic,
        "p_value": p_value
    }

## 4. Test Numeric Variables

In [8]:
numeric_tests = pd.DataFrame([
    numeric_test(stats_df, "tenure_months"),
    numeric_test(stats_df, "monthly_charge"),
    numeric_test(stats_df, "total_revenue"),
    numeric_test(stats_df, "satisfaction_score"),
    numeric_test(stats_df, "cltv")
]).sort_values("p_value")

numeric_tests

,variable,churned_mean,retained_mean,difference,t_statistic,p_value
3,satisfaction_score,1.736,3.772,-2.036,-93.210,0.000
0,tenure_months,17.979,41.042,-23.062,-41.417,0.000
2,total_revenue,"1,971.354","3,735.676","-1,764.323",-25.190,0.000
1,monthly_charge,74.441,62.976,11.465,15.731,0.000
4,cltv,"4,149.415","4,530.189",-380.775,-11.801,0.000


## 5. Categorical Test Function

In [9]:
def chi_square_test(data, column):
    contingency = pd.crosstab(data[column], data["churn_flag"])
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

    return {
        "variable": column,
        "chi2": chi2,
        "p_value": p_value,
        "degrees_of_freedom": dof,
        "categories": contingency.shape[0]
    }

## 6. Test Categorical Variables

In [10]:
categorical_tests = pd.DataFrame([
    chi_square_test(stats_df, "contract"),
    chi_square_test(stats_df, "payment_method"),
    chi_square_test(stats_df, "internet_type"),
    chi_square_test(stats_df, "premium_tech_support"),
    chi_square_test(stats_df, "gender")
]).sort_values("p_value")

categorical_tests

,variable,chi2,p_value,degrees_of_freedom,categories
0,contract,"1,695.694",0.000,2,3
2,internet_type,590.438,0.000,3,4
1,payment_method,321.478,0.000,2,3
3,premium_tech_support,231.668,0.000,1,2
4,gender,0.240,0.624,1,2


## 7. Contract Churn Pattern

In [11]:
contract_rates = (
    stats_df.groupby("contract")["churn_flag"]
    .mean()
    .mul(100)
    .round(2)
    .rename("churn_rate_pct")
    .reset_index()
)

contract_rates

,contract,churn_rate_pct
0,Month-to-Month,51.690
1,One Year,10.880
2,Two Year,2.580


## 8. Payment Method Churn Pattern

In [12]:
payment_rates = (
    stats_df.groupby("payment_method")["churn_flag"]
    .mean()
    .mul(100)
    .round(2)
    .rename("churn_rate_pct")
    .reset_index()
    .sort_values("churn_rate_pct", ascending=False)
)

payment_rates

,payment_method,churn_rate_pct
2,Mailed Check,41.400
0,Bank Withdrawal,35.650
1,Credit Card,15.810


## 9. Internet Type Churn Pattern

In [13]:
internet_rates = (
    stats_df.groupby("internet_type")["churn_flag"]
    .mean()
    .mul(100)
    .round(2)
    .rename("churn_rate_pct")
    .reset_index()
    .sort_values("churn_rate_pct", ascending=False)
)

internet_rates

,internet_type,churn_rate_pct
2,Fiber Optic,42.130
0,Cable,27.520
1,DSL,19.970
3,No Internet Service,8.410


## 10. Statistical Significance Summary

In [14]:
numeric_tests["significant_at_0_05"] = numeric_tests["p_value"] < 0.05
categorical_tests["significant_at_0_05"] = categorical_tests["p_value"] < 0.05

numeric_tests, categorical_tests

(             variable  churned_mean  retained_mean  difference  t_statistic  \
 3  satisfaction_score         1.736          3.772      -2.036      -93.210   
 0       tenure_months        17.979         41.042     -23.062      -41.417   
 2       total_revenue     1,971.354      3,735.676  -1,764.323      -25.190   
 1      monthly_charge        74.441         62.976      11.465       15.731   
 4                cltv     4,149.415      4,530.189    -380.775      -11.801   
 
    p_value  significant_at_0_05  
 3    0.000                 True  
 0    0.000                 True  
 2    0.000                 True  
 1    0.000                 True  
 4    0.000                 True  ,
                variable      chi2  p_value  degrees_of_freedom  categories  \
 0              contract 1,695.694    0.000                   2           3   
 2         internet_type   590.438    0.000                   3           4   
 1        payment_method   321.478    0.000                   2       

## 11. Interpret the Tests Carefully

I use p-values as evidence against the null hypothesis under the test assumptions.

I do not interpret statistical significance as proof that a variable causes churn.

I also consider:

- effect magnitude
- business relevance
- population size
- consistency with EDA
- potential leakage

## 12. Leakage Review

I exclude direct outcome fields from predictive features:

- `churn_label`
- `churn_category`
- `churn_reason`

I also treat `churn_score` and `CLTV` cautiously because the source does not document when or how
those fields were generated relative to the churn outcome.

If their construction uses post-outcome information, they would create leakage. I therefore keep
them out of the baseline predictive feature set until that timing is established.

## 13. Multiple Testing Awareness

I am testing several variables, so I avoid treating every p-value below 0.05 as a standalone
business discovery.

The results are used as evidence for a broader analytical story and should be checked for practical
importance and model consistency.

# Phase 4.2 Conclusion

I have formalized the strongest descriptive relationships into statistical tests and documented
the interpretation and leakage rules that will govern the predictive models.

**Next:** baseline churn modeling.